# Predict Individual Activation Peak

This notebook implements a two-stage analysis pipeline for spatially normalizing and decoding task-related activation peaks in the VMPFC.

In the second stage, we use these normalized voxel-wise labels to train a k-Nearest Neighbors (KNN) classifier. Hyperparameters (i.e., the number of neighbors) are optimized via a cross-validated grid search, and model performance is evaluated using a permutation test that repeatedly shuffles task labels to estimate the null distribution of classification accuracy. This approach allows us to assess whether local patterns of activity within the MPFC reliably distinguish between different psychological domains, and to visualize how these functional representations are organized across the left and right hemispheres.

In [1]:
from pathlib import Path
import numpy as np
from time import time
import scipy.io as sio
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
from sklearn.utils import shuffle
import joblib
from scipy.sparse import eye as sparse_eye
from tqdm import trange

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from utils import calculate_laplacian_matrix, calculate_Sn_new
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold


DATA_PATH = Path('../data')
RESULT_PATH = Path('../results')
BRAIN_HEMISPHERE_LIST = ['left', 'right']
LAMBDA_REG = 1e-1 # # balance smooth level
RADIUS = 2.5 # smooth radius
MARGIN = 0.7 # gap between voxels grids

# Model Training: Fit KNN

In this section, we train the K-Nearest Neighbors (KNN) classifier using 5-fold cross-validation. For each fold, the pipeline performs the following steps:
1. **Spatial Normalization:** Computes smoothed, voxel-wise domain labels for the training set using a Laplacian matrix and sparse linear equation solving.
2. **Hyperparameter Tuning:** Conducts a Grid Search to identify the optimal number of neighbors (`n_neighbors`).
3. **Evaluation:** Tests the best estimator on the unseen test dataset and saves the accuracy metrics and model weights for later evaluation.

In [3]:
raw_data_path = DATA_PATH / 'individual/raw_data.dict.pkl'
raw_data_dict = joblib.load(raw_data_path)
train_test_idx_path = DATA_PATH / 'individual/train_test_index.dict.pkl'
train_test_idx_dict = joblib.load(train_test_idx_path)

model = KNeighborsClassifier(algorithm='kd_tree', n_jobs=32)
param_grid = {'n_neighbors': [1000, 2000, 3000, 5000, 8000]}
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=5)

In [4]:
for brain_hemisphere in BRAIN_HEMISPHERE_LIST:
    acc_list = []
    peak_wide = raw_data_dict[f'{brain_hemisphere}_peak']
    for fold_i in range(5):
        (train_idx, test_idx) = train_test_idx_dict[(brain_hemisphere, fold_i)]
        print(f'Now processing: {brain_hemisphere} - fold{fold_i}')

        # 1. Get X train
        X_train = raw_data_dict[f'{brain_hemisphere}_MNI']

        # 2. Get y train
        ## load mni grid data
        mni_loc = raw_data_dict[f'{brain_hemisphere}_MNI']
        print(f'N points in MNI: {len(mni_loc)}')
        peak_loc = peak_wide[train_idx, :]
        print(f'N subjects: {len(peak_loc)}')
        print('Now calculating the Laplacian Matrix, might take a while...')

        L = raw_data_dict[f'{brain_hemisphere}_laplacian_matrix']
        ## Spatial Normalization for each voxel
        Sn = calculate_Sn_new(peak_loc, mni_loc, RADIUS)
        Sn = csc_matrix(Sn)
        I = sparse_eye(L.shape[0], format='csc')
        norm_matrix = LAMBDA_REG * spsolve(L + LAMBDA_REG * I, Sn)
        ## predict task labels
        ## +1: Python index starts from 0, while MATLAB start from 1
        y_train = np.argmax(norm_matrix.toarray(), axis=1) + 1

        # 3. Get X test and y test
        peak_wide_test = peak_wide[test_idx, :]
        X_test = np.vstack([peak_wide_test[:, 0:3], peak_wide_test[:, 3:6], peak_wide_test[:, 6:9]])
        y_test = np.repeat(np.arange(1, 4), peak_wide_test.shape[0])

        print(f'X train: {X_train.shape}, y train: {y_train.shape}')
        print(f'X test: {X_test.shape} y test: {y_test.shape}')

        # 4. Fit Model
        # hyperparameter tuning
        grid_search.fit(X_train, y_train)
        print("Best parameters found: ", grid_search.best_params_)
        y_predict = grid_search.best_estimator_.predict(X_test)
        acc = (y_predict == y_test).mean()
        print(' Accuracy:', acc)
        acc_list.append(acc)
        fitted_grid_search_file_path = RESULT_PATH / f'individual/{brain_hemisphere}_fold{fold_i}_KNN_GridSearch.pkl'
        joblib.dump(grid_search, fitted_grid_search_file_path)

    acc_list_file = RESULT_PATH / f'{brain_hemisphere}_KNN_acc.pkl'
    joblib.dump(acc_list, acc_list_file)

Now processing: left - fold0
N points in MNI: 52809
N subjects: 533
Now calculating the Laplacian Matrix, might take a while...
X train: (52809, 3), y train: (52809,)
X test: (402, 3) y test: (402,)


/home/guoqiu/.conda/envs/vmpfc_general/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found:  {'n_neighbors': 3000}
 Accuracy: 0.6840796019900498
Now processing: left - fold1
N points in MNI: 52809
N subjects: 533
Now calculating the Laplacian Matrix, might take a while...
X train: (52809, 3), y train: (52809,)
X test: (402, 3) y test: (402,)


/home/guoqiu/.conda/envs/vmpfc_general/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found:  {'n_neighbors': 2000}
 Accuracy: 0.6592039800995025
Now processing: left - fold2
N points in MNI: 52809
N subjects: 534
Now calculating the Laplacian Matrix, might take a while...
X train: (52809, 3), y train: (52809,)
X test: (399, 3) y test: (399,)


...

Best parameters found:  {'n_neighbors': 3000}
 Accuracy: 0.7192982456140351
Now processing: right - fold3
N points in MNI: 54072
N subjects: 534
Now calculating the Laplacian Matrix, might take a while...
X train: (54072, 3), y train: (54072,)
X test: (399, 3) y test: (399,)


/home/guoqiu/.conda/envs/vmpfc_general/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found:  {'n_neighbors': 3000}
 Accuracy: 0.6892230576441103
Now processing: right - fold4
N points in MNI: 54072
N subjects: 534
Now calculating the Laplacian Matrix, might take a while...
X train: (54072, 3), y train: (54072,)
X test: (399, 3) y test: (399,)


/home/guoqiu/.conda/envs/vmpfc_general/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found:  {'n_neighbors': 3000}
 Accuracy: 0.681704260651629


# Statistical Validation: Permutation Test

To establish the statistical significance of our model's performance, we conduct a permutation test.

First, we retrieve the optimal hyperparameters identified during the previous cross-validation step. Then, we retrain the KNN classifier on datasets where the training labels have been randomly shuffled. By calculating the accuracy of these shuffled models, we generate a null distribution of classification accuracies, which allows us to determine if our true model's predictive power is significantly better than chance.

In [7]:
hyper_parameter_dict = dict()
for brain_hemisphere in BRAIN_HEMISPHERE_LIST:
    hyper_parameter_list = []
    for fold_i in range(5):
        fitted_grid_search_file_path = RESULT_PATH / f'{brain_hemisphere}_fold{fold_i}_KNN_GridSearch.pkl'
        fitted_grid_search = joblib.load(fitted_grid_search_file_path)
        hyper_parameter_list.append(fitted_grid_search.best_params_['n_neighbors'])
    hyper_parameter_dict[brain_hemisphere] = hyper_parameter_list
hyper_parameter_dict

{'left': [3000, 2000, 5000, 2000, 3000],
 'right': [2000, 3000, 3000, 3000, 3000]}

In [14]:
for brain_hemisphere in ['left', 'right']:
    acc_list_file = RESULT_PATH / f'{brain_hemisphere}_KNN_permutation_acc.pkl'
    if acc_list_file.exists():
        continue
    permutation_data_path = DATA_PATH / f'{brain_hemisphere}_permutation.list.pkl'
    permutation_data_dict_list = joblib.load(permutation_data_path)
    permutation_acc_list = []
    for permutation_data_dict in tqdm(permutation_data_dict_list):
        fold_i = permutation_data_dict['fold_i']
        model.n_neighbors ==hyper_parameter_dict[brain_hemisphere][fold_i]
        X_train = raw_data_dict[f'{brain_hemisphere}_MNI']
        model.fit(X_train, permutation_data_dict['y_train'])
        y_predict = model.predict(permutation_data_dict['X_test'])
        acc = (y_predict == permutation_data_dict['y_test']).mean()
        permutation_acc_list.append(acc)
    
    joblib.dump(permutation_acc_list, acc_list_file)